# SGD with momentum
The update by hand, momentum on a narrow valley and on a curve with a local minimum, the effect of beta,
and plain SGD against SGD with momentum on MNIST.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## The update rule, two steps by hand
L(w) = w^2 / 2, so the gradient is w. Start w = -10, learning rate 0.1, beta = 0.9.

In [ ]:
def momentum_path(grad, w0, eta, beta, steps):
    """v_t = beta * v_{t-1} + eta * grad(w_t);  w_{t+1} = w_t - v_t.  beta = 0 is plain gradient descent."""
    w, v, path = np.array(w0, dtype=float), np.zeros_like(np.array(w0, dtype=float)), [np.array(w0, dtype=float)]
    for _ in range(steps):
        v = beta * v + eta * grad(w)
        w = w - v
        path.append(w.copy())
    return np.array(path)

print("momentum, w0..w2:        ", momentum_path(lambda w: w, -10.0, 0.1, 0.9, 2))
print("gradient descent, w0..w2:", momentum_path(lambda w: w, -10.0, 0.1, 0.0, 2))

## Steps grow while the gradients agree: the terminal step size is eta * g / (1 - beta)
A constant gradient g = 1 (a tilted plane).

In [ ]:
for beta in (0.5, 0.9, 0.99):
    v = 0.0
    sizes = []
    for t in range(2000):
        v = beta * v + 0.1 * 1.0
        sizes.append(v)
    print(f"beta {beta}: step 1 = {sizes[0]:.3f}, step 10 = {sizes[9]:.3f}, after many steps = {sizes[-1]:.3f}, "
          f"eta/(1-beta) = {0.1 / (1 - beta):.3f}")

## The narrow valley L = (w1^2 + 100 w2^2) / 2 from (-10, 0.4)
Steps until the loss is below 0.01.

In [ ]:
K = 100
valley_grad = lambda w: np.array([w[0], K * w[1]])
valley_loss = lambda P: 0.5 * (P[:, 0] ** 2 + K * P[:, 1] ** 2)

def steps_to(P, tol=0.01):
    below = valley_loss(P) < tol
    return int(np.argmax(below)) if below.any() else None

for eta, beta in ((0.01, 0.0), (0.019, 0.0), (0.01, 0.5), (0.01, 0.8), (0.01, 0.9)):
    P = momentum_path(valley_grad, [-10.0, 0.4], eta, beta, 3000)
    print(f"eta {eta:<5} beta {beta}: {steps_to(P)} steps")
P_gd = momentum_path(valley_grad, [-10.0, 0.4], 0.01, 0.0, 60)
P_m = momentum_path(valley_grad, [-10.0, 0.4], 0.01, 0.9, 60)
print("after 20 steps, w1: gradient descent", P_gd[20, 0].round(2), " momentum", P_m[20, 0].round(2))
print("momentum overshoots to w1 =", P_m[:, 0].max().round(2), "past the minimum at 0")

## The effect of beta on L(w) = w^2 / 2, learning rate 0.1, 100 steps

In [ ]:
for beta in (0.0, 0.5, 0.9, 1.0):
    P = momentum_path(lambda w: w, -10.0, 0.1, beta, 100)
    crossings = int(np.sum(np.diff(np.sign(P)) != 0))
    print(f"beta {beta}: w after 100 steps = {P[-1]:8.4f}, crosses the minimum {crossings} times, "
          f"largest |w| in steps 80-100 = {np.abs(P[80:]).max():.3f}")

## A curve with a local minimum: L(w) = (w^2 - 4)^2 / 8 - 0.6 w, start w = -3, learning rate 0.05

In [ ]:
f = lambda w: (w ** 2 - 4) ** 2 / 8 - 0.6 * w
df = lambda w: w * (w ** 2 - 4) / 2 - 0.6
for beta in (0.0, 0.5, 0.9):
    P = momentum_path(df, -3.0, 0.05, beta, 300)
    print(f"beta {beta}: ends at w = {P[-1]:.2f} (loss {f(P[-1]):.2f}), furthest right {P.max():.2f}")
print("local minimum near -1.83 (loss", round(f(-1.8305), 2), "), global near 2.14 (loss", round(f(2.135), 2), ")")

## MNIST: plain SGD against SGD with momentum
10,000 training images, the 10,000 test images for validation. Same network, same learning rate 0.01,
batch size 64, 20 epochs; only momentum changes (0 or 0.9). Three seeds each.

In [ ]:
(x, y), (x_test, y_test) = keras.datasets.mnist.load_data()
x = x[:10000].reshape(-1, 784).astype("float32") / 255     # pixels to 0..1
y = y[:10000]
x_test = x_test.reshape(-1, 784).astype("float32") / 255

def train(momentum, seed, epochs=20):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([keras.Input(shape=(784,)),
                              keras.layers.Dense(128, activation="relu"),
                              keras.layers.Dense(64, activation="relu"),
                              keras.layers.Dense(10, activation="softmax")])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=momentum),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model.fit(x, y, epochs=epochs, batch_size=64, validation_data=(x_test, y_test), verbose=0).history

rows = []
for momentum in (0.0, 0.9):
    for seed in range(3):
        h = train(momentum, seed)
        for e in range(len(h["loss"])):
            rows.append(dict(momentum=momentum, seed=seed, epoch=e + 1, loss=h["loss"][e],
                             val_accuracy=h["val_accuracy"][e]))
        print(f"momentum {momentum} seed {seed}: training loss after 20 epochs {h['loss'][-1]:.4f}, "
              f"validation accuracy {h['val_accuracy'][-1]:.4f}")
runs = pd.DataFrame(rows)
runs.to_csv("data/mnist_loss.csv", index=False)

In [ ]:
mean = runs.groupby(["momentum", "epoch"])[["loss", "val_accuracy"]].mean().unstack(0)
print(mean.loc[[1, 2, 5, 10, 20]].round(4))
sgd_final = mean[("loss", 0.0)].iloc[-1]
first = int((mean[("loss", 0.9)] <= sgd_final).idxmax())
print(f"plain SGD's loss after 20 epochs: {sgd_final:.4f}; momentum gets there by epoch {first}")